# Probing Classifiers

**Survey Reference:** Section 9.6 - Probing Classifiers: Capabilities, Critiques, and Causal Limitations

## Overview

**Probing classifiers** (probes) are trained to predict properties from neural network representations. If a linear probe achieves high accuracy, the representation likely encodes that property in a linearly accessible way.

### Key Questions

- What information is **encoded** in representations?
- Is the information **used** by the model? (Causal question)
- How should we interpret probe results?

## Learning Objectives

1. Train linear probes on model representations
2. Understand the correlation vs. causation problem
3. Implement control tasks for probe validation
4. Use amnesic probing for causal insights

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Basic Probing Setup

In [ ]:
# Load model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

def get_representation(model, tokenizer, text: str, layer_idx: int):
    """Extract representation from a specific layer."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    # Return last position's hidden state
    return outputs.hidden_states[layer_idx][0, -1, :].cpu().numpy()

## 2. Example: Sentiment Probe

Train a probe to detect sentiment from movie review representations.

In [ ]:
# Create sentiment dataset
positive_texts = [
    "This movie was absolutely wonderful and amazing.",
    "I loved every moment of this brilliant film.",
    "An outstanding masterpiece that moved me to tears.",
    "The best movie I have ever seen in my life.",
    "Fantastic acting and a beautiful story.",
]

negative_texts = [
    "This movie was absolutely terrible and boring.",
    "I hated every moment of this awful film.",
    "A disappointing disaster that wasted my time.",
    "The worst movie I have ever seen in my life.",
    "Terrible acting and a confusing story.",
]

# Labels: 1 = positive, 0 = negative
texts = positive_texts + negative_texts
labels = [1] * len(positive_texts) + [0] * len(negative_texts)

In [ ]:
def collect_representations(model, tokenizer, texts, layer_idx):
    """Collect representations for a list of texts."""
    representations = []
    for text in texts:
        rep = get_representation(model, tokenizer, text, layer_idx)
        representations.append(rep)
    return np.stack(representations)

# Collect from middle layer
layer_idx = model.config.n_layer // 2
X = collect_representations(model, tokenizer, texts, layer_idx)
y = np.array(labels)

print(f"Representations shape: {X.shape}")

In [ ]:
def train_probe(X, y, test_size=0.2):
    """Train a linear probe and evaluate."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    probe = LogisticRegression(max_iter=1000)
    probe.fit(X_train, y_train)
    
    train_acc = probe.score(X_train, y_train)
    test_acc = probe.score(X_test, y_test)
    
    return probe, train_acc, test_acc

probe, train_acc, test_acc = train_probe(X, y)
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## 3. Layer-wise Probing

Probe at each layer to see where information appears.

In [ ]:
def layerwise_probing(model, tokenizer, texts, labels):
    """Probe at each layer to track where information appears."""
    n_layers = model.config.n_layer + 1  # +1 for embedding
    y = np.array(labels)
    
    accuracies = []
    for layer_idx in range(n_layers):
        X = collect_representations(model, tokenizer, texts, layer_idx)
        _, _, test_acc = train_probe(X, y)
        accuracies.append(test_acc)
    
    return accuracies

# This would take a while with more data
# accuracies = layerwise_probing(model, tokenizer, texts, labels)
# plt.plot(accuracies)
# plt.xlabel('Layer')
# plt.ylabel('Probe Accuracy')
# plt.show()

## 4. The Causal Problem

**High probe accuracy ≠ the model uses this information!**

A probe might decode information that:
- Is **present** but not used
- Is a **byproduct** of other computations
- Is **extractable** only by the probe's complexity

In [ ]:
# Demonstration: Probes can find spurious correlations
# Even random labels might be partially learnable with high-dimensional data!

np.random.seed(42)
random_labels = np.random.randint(0, 2, size=len(texts))

_, _, random_acc = train_probe(X, random_labels)
print(f"Accuracy on random labels: {random_acc:.4f}")
print("(Should be ~0.5, but high dimensions can lead to overfitting)")

## 5. Control Tasks (Selectivity)

**Control tasks** (Hewitt & Liang, 2019) test if probing results are meaningful:

1. Train probe on real labels → accuracy $A_{real}$
2. Train probe on random labels → accuracy $A_{random}$
3. **Selectivity** = $A_{real} - A_{random}$

High selectivity indicates the information is specifically encoded.

In [ ]:
def compute_selectivity(X, y_real, n_random_trials=5):
    """
    Compute selectivity: real accuracy - random baseline.
    """
    # Real accuracy
    _, _, real_acc = train_probe(X, y_real)
    
    # Random baseline (average over trials)
    random_accs = []
    for _ in range(n_random_trials):
        y_random = np.random.permutation(y_real)  # Shuffle labels
        _, _, rand_acc = train_probe(X, y_random)
        random_accs.append(rand_acc)
    
    random_baseline = np.mean(random_accs)
    selectivity = real_acc - random_baseline
    
    print(f"Real accuracy: {real_acc:.4f}")
    print(f"Random baseline: {random_baseline:.4f}")
    print(f"Selectivity: {selectivity:.4f}")
    
    return selectivity

selectivity = compute_selectivity(X, y)

## 6. Amnesic Probing

**Amnesic probing** (Elazar et al., 2021) tests if information is **causally used**:

1. Project out the probed information from representations
2. Check if model performance on the task degrades

If performance drops, the information was causally important.

In [ ]:
def amnesic_projection(X, y, probe):
    """
    Project out the direction used by the probe.
    
    Returns representations with the probed information removed.
    """
    # Get probe direction (for logistic regression)
    direction = probe.coef_[0]  # [hidden_size]
    direction = direction / np.linalg.norm(direction)
    
    # Project out this direction from X
    # X_projected = X - (X · direction) * direction
    projections = (X @ direction).reshape(-1, 1)
    X_amnesic = X - projections @ direction.reshape(1, -1)
    
    return X_amnesic

# Project out sentiment direction
X_amnesic = amnesic_projection(X, y, probe)

# Train new probe on amnesic representations
_, _, amnesic_acc = train_probe(X_amnesic, y)
print(f"Original accuracy: {test_acc:.4f}")
print(f"After projecting out probe direction: {amnesic_acc:.4f}")

## 7. Probe Complexity Considerations

The complexity of the probe matters:

| Probe Type | What it shows |
|------------|---------------|
| Linear | Information is linearly accessible |
| MLP | Information exists (but may require computation) |
| Complex | Probe might be doing the computation itself |

In [ ]:
class MLPProbe(nn.Module):
    """Nonlinear probe with one hidden layer."""
    
    def __init__(self, input_dim, hidden_dim, n_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, n_classes)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Compare linear vs. MLP probe to see if information is linearly accessible

## 8. Best Practices for Probing Studies

1. **Use control tasks** to measure selectivity
2. **Prefer linear probes** unless you have a specific reason
3. **Report confidence intervals** over multiple random seeds
4. **Consider amnesic probing** for causal claims
5. **Compare layers** to understand information flow

In [ ]:
# TODO: Full probing study with proper validation

## Key Takeaways

1. **Probes reveal what's encoded,** not necessarily what's used
2. **Control tasks** distinguish meaningful from spurious results
3. **Amnesic probing** provides causal insights
4. **Probe complexity** affects interpretation
5. **Layer-wise analysis** shows where information appears

## Next Steps

→ **07_representation_analysis/01_model_diffing.ipynb**: Compare representations across models